## **다중  LLM 활용**
-  순차 Multi-LLM + GPT 종합 구조

> **한눈에 보기**
> - 다중 LLM 두 가지 패턴: **병렬**(GPT·Gemini·EXAONE에 동시에 묻고 GPT가 종합) / **순차**(Gemini 설계 → EXAONE 구현 → GPT 검증)
> - **ReAct**: Thought(생각) → Action(도구 호출) → Observation(결과)을 반복하는 에이전트
> - 실패 결과를 보고 다른 행동을 고르는 재시도 루프, 돈이 나가는 행동만 사람이 승인하는 Human-in-the-loop
> - 정리 문서: [다중LLM과ReAct.md](다중LLM과ReAct.md)
> - 슬라이드 이미지는 자리만 표시, 📝 정리 셀은 실행 결과를 보고 직접 정리

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [4]:
# 주의: google-genai의 client.models에는 send_message가 없다 (채팅 객체 client.chats.create(...)에 있는 메서드).
# 그래서 ask_gemini는 매번 except로 빠져 'Gemini 에러: ...' 문자열을 돌려줬다.
# 올바른 호출은 12번 셀처럼 google_client.models.generate_content(model=..., contents=...)
import os
from openai import OpenAI
from google import genai

# 각 LLM의 클라이언트 초기화 (환경 변수 기반)
# 1. OpenAI (gpt-4o-mini)
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 2. Google (gemini-3.5-flash-lite) - 최신 공식 google-genai SDK 규격
google_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# 3. 로컬 오픈소스 모델 (Ollama - EXAONE 3.5 또는 Llama 3.1)
# Ollama는 OpenAI의 API 규격과 100% 호환되므로 OpenAI 클라이언트를 재활용하면 가장 심플합니다.
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # 로컬이므로 아무 문자열이나 넣어도 됩니다.
)

def ask_gpt(prompt: str) -> str:
    """OpenAI gpt-4o-mini 호출 (최신 Responses API 구조)"""
    try:
        response = openai_client.responses.create(
            model="gpt-4o-mini",
            instructions="당신은 분석적이고 객관적인 IT 전문가입니다.",
            input=prompt
        )
        return response.output_text
    except Exception as e:
        return f"OpenAI 에러: {e}"

def ask_gemini(prompt: str) -> str:
    """Google gemini-3.5-flash-lite 호출 (최신 2.0 SDK 구조)"""
    try:
        response = google_client.models.send_message(
            model='gemini-3.5-flash-lite',
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"Gemini 에러: {e}"

def ask_ollama(prompt: str) -> str:
    """로컬 Ollama 호출 (Llama3.1 또는 EXAONE-3.5 선택)"""
    try:
        # 사용 중인 로컬 모델명으로 변경하세요 (예: 'exaone3.5:7b' 또는 'llama3.1')
        model_name = "exaone3.5" 
        
        response = ollama_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Ollama 에러 (모델 확인 필요): {e}"

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
from concurrent.futures import ThreadPoolExecutor

def main():
    user_question = "초보 개발자의 첫 웹 백엔드 스택으로 Python FastAPI와 Java Spring Boot 중 무엇을 더 추천하나요?"

    print(f"❓ 공동 질문: '{user_question}'\n")
    print("==============================================")
    print("🔄 1단계: 3개의 다중 LLM에게 동시에 의견 묻는 중...")
    print("==============================================\n")

    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            "GPT": executor.submit(ask_gpt, user_question),
            "Gemini": executor.submit(ask_gemini, user_question),
            "Ollama": executor.submit(ask_ollama, user_question),
        }

        gpt_answer = futures["GPT"].result()
        print("🤖 [OpenAI gpt-4o-mini 답변 완료]\n")

        gemini_answer = futures["Gemini"].result()
        print("♊ [Google gemini-3.5-flash-lite 답변 완료]\n")

        ollama_answer = futures["Ollama"].result()
        print("🦙 [Ollama 로컬 모델 답변 완료]\n")

    synthesis_prompt = f"""
    당신은 수석 아키텍처이자 마스터 멘토입니다. 
    아래는 동일한 질문에 대해 3개의 서로 다른 AI 모델(GPT, Gemini, 오픈소스 모델)이 내놓은 답변입니다.

    [질문]
    {user_question}

    ---
    [AI 1: GPT의 의견]
    {gpt_answer}

    ---
    [AI 2: Gemini의 의견]
    {gemini_answer}

    ---
    [AI 3: 오픈소스 모델의 의견]
    {ollama_answer}
    ---

    위 3가지 시각을 종합 분석하여, 공통적인 핵심 권장 사항을 추려내고, 
    각 AI의 답변 스타일 차이점을 가볍게 평가하는 '최종 종합 가이드 리포트'를 깔끔한 마크다운 양식으로 작성해 주세요.
    """

    print("==============================================")
    print("🔄 2단계: 수집된 의견들을 종합 요약하는 중 (by gpt-4o-mini)...")
    print("==============================================\n")

    final_report = ask_gpt(synthesis_prompt)

    print("✨ [최종 다중 LLM 취합 리포트] ✨\n")
    print(final_report)

In [6]:
main()

❓ 공동 질문: '초보 개발자의 첫 웹 백엔드 스택으로 Python FastAPI와 Java Spring Boot 중 무엇을 더 추천하나요?'

🔄 1단계: 3개의 다중 LLM에게 동시에 의견 묻는 중...

🤖 [OpenAI gpt-4o-mini 답변 완료]

♊ [Google gemini-3.5-flash-lite 답변 완료]

🦙 [Ollama 로컬 모델 답변 완료]

🔄 2단계: 수집된 의견들을 종합 요약하는 중 (by gpt-4o-mini)...

✨ [최종 다중 LLM 취합 리포트] ✨

# 최종 종합 가이드 리포트

## 웹 백엔드 스택 추천: Python FastAPI vs Java Spring Boot

### 공통적인 핵심 권장 사항

1. **Python FastAPI 추천**:
   - **쉬운 학습 곡선**: Python의 간결하고 직관적인 문법 덕분에 초보자에게 적합합니다.
   - **빠른 개발**: FastAPI는 비동기 프로그래밍 지원 및 자동 문서화 기능 덕에 신속한 프로토타입 작성이 가능합니다.
   - **활발한 커뮤니티**: Python 생태계의 커뮤니티가 활성화되어 있어 지원 및 자료를 구하기 용이합니다.

2. **Java Spring Boot 고려 사항**:
   - **엔터프라이즈 솔루션**: 대규모 시스템이나 복잡한 아키텍처를 고려하는 경우 적합합니다.
   - **성숙한 생태계**: 오랜 역사와 방대한 자료를 바탕으로 안정성이 뛰어납니다.
   - **진입 장벽**: 설정이 복잡하고 비교적 높은 학습 곡선이 있을 수 있습니다.

3. **추가 학습 팁**:
   - 웹 개발의 기본기(HTML, CSS, JavaScript 등)를 다지는 것이 중요합니다.
   - 실습 중심의 학습을 권장하며, 작은 프로젝트부터 시작해 실력을 쌓아 나가는 것이 효과적입니다.

---

### AI 답변 스타일 차이 평가

- **AI 1: GPT**  
  - **스타일**: 체계적이고 균형 잡힌 분석. 장단점을 명확하게 분리하여 

**📝 정리** — 세 모델에 같은 질문을 동시에 보내고 GPT가 종합했다. 리포트가 Gemini를 "기술적 오류로 인해 미비함"이라고 평가한 것은, 3번 셀의 `ask_gemini`가 없는 메서드(`models.send_message`)를 불러 **에러 문자열**을 답변 자리에 넣었기 때문이다. 종합하는 모델은 그것도 답변으로 받아 평가했다 → 여러 답을 합치기 전에 각 호출이 성공했는지 먼저 확인해야 한다.

```
with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            "GPT": executor.submit(ask_gpt, user_question),
            "Gemini": executor.submit(ask_gemini, user_question),
            "Ollama": executor.submit(ask_ollama, user_question),
        }

        gpt_answer = futures["GPT"].result()
        print("🤖 [OpenAI gpt-4o-mini 답변 완료]\n")

        gemini_answer = futures["Gemini"].result()
        print("♊ [Google gemini-3.5-flash-lite 답변 완료]\n")

        ollama_answer = futures["Ollama"].result()
        print("🦙 [Ollama 로컬 모델 답변 완료]\n")
```

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

**다중 LLM - 순차 파이프라인 패턴**

- gemini-3.5-flash-lite → **시스템 아키텍트** (고수준 설계/명세 작성에 강점)
- Ollama/exaone 3.5 (로컬) → **풀스택 개발자** (실제 코드 생성)
- gpt-4o-mini → ***시니어 QA 리뷰어** (엄격한 검증/판정)

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [7]:
import inspect
from openai import OpenAI
from google import genai

# 1. 2026년 기준 가장 최신 규격의 클라이언트 선언
# OpenAI Responses API용 클라이언트
openai_client = OpenAI() 

# Google 최신 공식 google-genai SDK용 클라이언트 (환경변수 GEMINI_API_KEY 자동 매핑)
google_client = genai.Client()

# 로컬 오픈소스용 클라이언트 (OpenAI SDK로 Ollama 최신 v1 표준 엔드포인트 연동)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# -------------------------------------------------------------
# Agent 1: Google Gemini 3.5 Flash Lite (역할: 시스템 아키텍트)
# -------------------------------------------------------------
def run_planner_agent(user_idea: str) -> str:
    print("▶️ [Agent 1] Gemini 아키텍트가 설계 명세서를 작성 중입니다...")
    
    prompt = inspect.cleandoc(f"""
        당신은 전설적인 시스템 아키텍트입니다. 
        사용자의 아이디어를 바탕으로, 개발자가 즉시 코딩에 착수할 수 있도록 구체적인 '기능 명세서'를 작성하세요.
        반드시 포함할 내용: 필요한 함수 목록, 입력/출력 데이터 타입, 예외 처리 기준.
        
        [사용자 아이디어]: {user_idea}
    """)
    
    # google-genai 최신 표준 메서드 호출
    response = google_client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
    )
    return response.text

# -------------------------------------------------------------
# Agent 2: Ollama / EXAONE 3.5 (역할: 풀스택 개발자)
# -------------------------------------------------------------
def run_developer_agent(spec_document: str) -> str:
    print("▶️ [Agent 2] 로컬 오픈소스 개발자가 코드를 구현 중입니다...")
    
    prompt = inspect.cleandoc(f"""
        당신은 천재 파이썬 개발자입니다. 
        전달받은 [기능 명세서]를 100% 반영하여 구동 가능한 깔끔한 파이썬 소스 코드를 작성하세요.
        설명은 생략하고 마크다운 코드 블록(```python ... ```) 안에 코드만 완벽하게 작성하세요.
        
        [기능 명세서]:
        {spec_document}
    """)
    
    # Ollama Chat Completion 표준 API 호출
    response = ollama_client.chat.completions.create(
        model="exaone3.5", 
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# -------------------------------------------------------------
# Agent 3: OpenAI gpt-4o-mini (역할: 시니어 QA 리뷰어)
# -------------------------------------------------------------
def run_reviewer_agent(original_spec: str, generated_code: str) -> str:
    print("▶️ [Agent 3] GPT 시니어 리뷰어가 완성된 코드를 최종 검증 중입니다...")
    
    user_prompt = inspect.cleandoc(f"""
        당신은 엄격한 시니어 개발자이자 QA 엔지니어입니다.
        [개발자가 작성한 코드]가 원래의 [기능 명세서] 규칙을 모두 준수했는지 교차 검증하세요.
        버그 가능성이나 개선 방향을 짚어주고, 최종 승인 여부(Pass/Fail)를 판정하세요.
        
        [원래 기능 명세서]:
        {original_spec}
        
        [개발자가 작성한 코드]:
        {generated_code}
    """)
    
    # 순수 OpenAI 최신 표준 Responses API 활용
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="당신은 엄격하고 논리적인 코드 검증 및 품질 리뷰 어시스턴트입니다.",
        input=user_prompt
    )
    return response.output_text


# -------------------------------------------------------------
# 🚀 메인 오케스트레이션 파이프라인
# -------------------------------------------------------------
def main():
    user_idea = "사용자가 회원가입할 때 비밀번호를 입력받아 안전하게 해싱(SHA-256)하고, 솔트(Salt)를 추가하여 검증까지 해주는 간단한 보안 모듈을 만들어줘."
    
    print(f"💡 [User Request]: {user_idea}\n")
    print("==================================================================")
    print("🤖 최신 API 기반 멀티 에이전트 파이프라인 가동")
    print("==================================================================\n")
    
    # 1단계: 설계 (Gemini)
    spec_result = run_planner_agent(user_idea)
    print("✅ [Gemini] 설계 명세서 작성 완료.\n")
    
    # 2단계: 개발 (Ollama - EXAONE 3.5)
    code_result = run_developer_agent(spec_result)
    print("✅ [Ollama] 소스 코드 빌드 완료.\n")
    print("--- [생성된 소스 코드] ---")
    print(code_result)
    print("\n")
    
    # 3단계: 검증 (OpenAI)
    final_review = run_reviewer_agent(spec_result, code_result)
    print("✅ [OpenAI] 최종 QA 및 코드 리뷰 완료.\n")
    
    print("==================================================================")
    print("✨ [최종 파이프라인 합작 결과 리포트]")
    print("==================================================================\n")
    print(final_review)

if __name__ == "__main__":
    main()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


💡 [User Request]: 사용자가 회원가입할 때 비밀번호를 입력받아 안전하게 해싱(SHA-256)하고, 솔트(Salt)를 추가하여 검증까지 해주는 간단한 보안 모듈을 만들어줘.

🤖 최신 API 기반 멀티 에이전트 파이프라인 가동

▶️ [Agent 1] Gemini 아키텍트가 설계 명세서를 작성 중입니다...
✅ [Gemini] 설계 명세서 작성 완료.

▶️ [Agent 2] 로컬 오픈소스 개발자가 코드를 구현 중입니다...
✅ [Ollama] 소스 코드 빌드 완료.

--- [생성된 소스 코드] ---
```python
import os
import hashlib
import base64

class AuthSecurityModule:
    def generateSalt(self):
        try:
            # 암호학적으로 안전한 랜덤 바이트 생성 (16 바이트 이상 권장)
            random_bytes = os.urandom(16)
            # Hex 인코딩
            return base64.b64encode(random_bytes).decode('utf-8')
        except Exception as e:
            raise CryptoException("난수 생성 실패: " + str(e))

    def hashPassword(self, password: str, salt: str) -> str:
        if not password or not salt:
            raise InvalidParameterException("비밀번호와 솔트가 필요합니다.")
        # 솔트와 비밀번호 결합 후 SHA-256 해시 생성
        hash_object = hashlib.sha256()
        hash_object.update((salt + password).encode('utf-8'))
        return hash_ob

**📝 정리** — 같은 파이프라인인데 결과가 달랐다. 내 실행에서는 리뷰어가 **Fail**을 줬다(명세와 다른 인코딩, `==`로 해시 비교 → 타이밍 공격에 취약, 키 스트레칭 없음). 강사님이 준 실행결과.pdf에서는 개발 단계가 `hmac.compare_digest`를 써서 **Pass**였다. LLM 결과물은 매번 달라지므로 마지막 검증 단계가 필요하다.

## **ReAct 루프**

**ReAct: Synergizing Reasoning and Acting in Language Models** 
(언어 모델에서의 추론과 행동의 시너지 창출)
- 2023년 ICLR(International Conference on Learning Representations)에 발표된 논문

- LLM이 **생각하기(Thought)** 와 **행동하기(Action)** 를 번갈아 하면서, 행동의 결과(Observation)를 다시 다음 생각에 반영합니다. 
- 이 세 가지가 한 사이클을 이루고, 모델은 매 사이클마다 "이미 관찰한 사실"을 근거로 다음 판단을 갱신합니다.

**다른 패러다임과 비교**

- Chain-of-Thought: 추론만 하고 외부 세계와 상호작용이 없습니다. 틀린 가정을 검증할 방법이 없어 환각에 취약합니다.
- Act-only: 추론 과정 없이 바로 도구를 호출합니다. 왜 그 도구를 골랐는지 설명이 안 되고, 복잡한 다단계 문제에서 방향을 잃기 쉽습니다.
- Plan-and-Execute: 처음에 전체 계획을 세우고 그대로 밀어붙입니다. 중간에 예상 못한 정보(예: "어 비가 오네")가 나와도 계획을 수정하기 어렵습니다.
- ReAct: 매 단계마다 관찰 결과를 보고 생각을 다시 합니다. 동적으로 계획을 바꿀 수 있어 환각이 줄어듭니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### ReAct 소스(1)

In [8]:
# calculator는 일부러 3번에 1번만 계산하고 나머지는 빈 문자열을 돌려준다 (불안정한 도구 흉내)
# eval()은 학습용. LLM이 만든 식을 그대로 실행하므로 실제 서비스에서는 ast로 허용한 연산만 계산한다 (-> ex1015 exam19)
# count는 빈 결과(실패)일 때만 늘어나서, 실패한 시도를 최대 5번까지 허용한다
"""ReAct 루프의 본질만 다루는 최소 예제"""
import json
from openai import OpenAI
import random

client = OpenAI()


def calculator(expression: str) -> str:
    """사칙연산 계산기 (학습용 eval → 실제 서비스에선 안전한 파서 권장)"""
    return str(eval(expression)) if random.randint(1, 3) == 1 else ""


tools = [{
    "type": "function",
    "name": "calculator",
    "description": "사칙연산 수식을 계산합니다.",
    "parameters": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}]


def run(question: str):
    messages = [{"role": "user", "content": question}]

    count = 1
    while count <= 5:  # 최대 5번까지 반복 (무한루프 방지)
        # ① 메시지 + 도구 목록을 모델에 보낸다
        response = client.responses.create(
            model="gpt-4o-mini",
            tools=tools,
            input=messages,
        )
        messages += response.output

        calls = [item for item in response.output if item.type == "function_call"]

        if not calls:                       # 도구 요청이 없다 = 최종 답변
            return response.output_text

        # ② 도구 호출 요청을 실행하고 결과를 다시 넣는다
        for call in calls:
            args = json.loads(call.arguments)
            result = calculator(**args)
            print(f"tool: {call.name}({args}) -> {result}")
            if result != "" :
                count -= 1

            messages.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result,
            }) 
        count += 1       

    return "최대 반복 횟수를 초과했습니다."

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [9]:
answer = run("153 곱하기 27은 얼마야? 계산기로 정확히 확인해줘.")
print("최종 답변:", answer)

tool: calculator({'expression': '153 * 27'}) -> 
tool: calculator({'expression': '153 * 27'}) -> 4131
최종 답변: 153 곱하기 27은 4131입니다.


In [10]:
answer = run("1에 2를 더하고 이 결과에 4를 곱하면 얼마야? 계산기로 정확히 확인해줘.")
print("최종 답변:", answer)

tool: calculator({'expression': '(1 + 2) * 4'}) -> 12
최종 답변: 1에 2를 더하고 그 결과에 4를 곱하면 12입니다.


**📝 정리** — 19번 셀에서 첫 계산이 빈 결과(`->` 뒤가 비어 있음)로 오자 모델이 같은 식으로 다시 호출해 4131을 얻었다. 결과(Observation)를 보고 다시 행동하는 것이 ReAct의 핵심이다. 20번은 `(1 + 2) * 4`처럼 식을 스스로 묶어 한 번에 성공했다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### ReAct 소스(2)

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [11]:
from openai import OpenAI
import json

MODEL = "gpt-4o-mini"
client = OpenAI()


# ------------------------------------------------------------
# 1. Toy Database
# ------------------------------------------------------------
# 외부 API를 쓰지 않고도 ReAct 흐름을 연습할 수 있도록
# 날씨, 장소, 비용 정보를 코드 안에 작게 넣어둡니다.

WEATHER_DB = {
    ("서울", "토요일"): {
        "condition": "비",
        "temperature_c": 24,
        "summary": "오후에 비가 올 가능성이 높습니다. 실내 활동을 추천합니다.",
    },
    ("서울", "일요일"): {
        "condition": "맑음",
        "temperature_c": 28,
        "summary": "대체로 맑고 야외 활동하기 좋습니다.",
    },
    ("부산", "토요일"): {
        "condition": "흐림",
        "temperature_c": 25,
        "summary": "흐리지만 비 가능성은 낮습니다.",
    },
}

PLACE_DB = [
    {
        "city": "서울",
        "name": "국립중앙박물관",
        "type": "문화",
        "indoor": True,
        "price_per_person": 0,
        "duration_hours": 2.5,
        "description": "비 오는 날에도 편하게 볼 수 있는 대형 실내 전시 공간입니다.",
    },
    {
        "city": "서울",
        "name": "성수동 카페거리",
        "type": "카페",
        "indoor": True,
        "price_per_person": 15000,
        "duration_hours": 2,
        "description": "카페와 편집숍을 함께 둘러보기 좋습니다.",
    },
    {
        "city": "서울",
        "name": "잠실 롯데월드몰",
        "type": "쇼핑",
        "indoor": True,
        "price_per_person": 20000,
        "duration_hours": 3,
        "description": "쇼핑, 식사, 전시를 한 번에 해결할 수 있는 실내 코스입니다.",
    },
    {
        "city": "서울",
        "name": "서울숲",
        "type": "산책",
        "indoor": False,
        "price_per_person": 0,
        "duration_hours": 2,
        "description": "날씨가 좋을 때 산책하기 좋은 야외 공간입니다.",
    },
    {
        "city": "서울",
        "name": "한강공원",
        "type": "산책",
        "indoor": False,
        "price_per_person": 10000,
        "duration_hours": 2,
        "description": "돗자리, 간식, 자전거 등을 즐길 수 있는 야외 코스입니다.",
    },
]


# ------------------------------------------------------------
# 2. Real Python Tools
# ------------------------------------------------------------
# 모델이 직접 실행하는 것이 아니라,
# 모델이 "이 함수를 호출해줘"라고 요청하면
# 우리가 Python 코드에서 실행해주는 구조입니다.

def get_weather(city: str, day: str) -> dict:
    """도시와 요일에 대한 날씨 정보를 가져옵니다."""
    result = WEATHER_DB.get((city, day))
    if result is None:
        return {
            "city": city,
            "day": day,
            "error": "해당 도시/요일의 날씨 정보가 없습니다.",
        }

    return {
        "city": city,
        "day": day,
        **result,
    }


def search_places(city: str, indoor_required: bool, preference: str) -> dict:
    """조건에 맞는 장소 후보를 검색합니다."""
    candidates = []

    for place in PLACE_DB:
        if place["city"] != city:
            continue

        if indoor_required and not place["indoor"]:
            continue

        # preference는 너무 엄격하게 필터링하지 않고,
        # 타입 또는 설명에 포함되면 점수를 조금 높이는 방식으로 씁니다.
        score = 0
        if preference in place["type"]:
            score += 2
        if preference in place["description"]:
            score += 1
        if place["indoor"]:
            score += 1

        candidates.append({
            **place,
            "match_score": score,
        })

    candidates.sort(key=lambda x: x["match_score"], reverse=True)

    return {
        "city": city,
        "indoor_required": indoor_required,
        "preference": preference,
        "places": candidates[:5],
    }


def estimate_total_cost(place_name: str, people: int) -> dict:
    """장소 이름과 인원수를 기반으로 총 예상 비용을 계산합니다."""
    for place in PLACE_DB:
        if place["name"] == place_name:
            total = place["price_per_person"] * people
            return {
                "place_name": place_name,
                "people": people,
                "price_per_person": place["price_per_person"],
                "total_cost": total,
                "currency": "KRW",
            }

    return {
        "place_name": place_name,
        "error": "장소를 찾을 수 없습니다.",
    }


# ------------------------------------------------------------
# 3. Tool Schemas for Responses API
# ------------------------------------------------------------
# Responses API의 function calling에서는 tools에 함수 스키마를 넣습니다.
# strict=True를 쓰려면 모든 properties를 required에 넣고,
# additionalProperties=False를 지정하는 것이 좋습니다.

TOOLS = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시와 요일을 입력받아 날씨를 확인합니다. 비가 오는지 판단할 때 사용합니다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "도시 이름. 예: 서울, 부산",
                },
                "day": {
                    "type": "string",
                    "description": "요일. 예: 토요일, 일요일",
                },
            },
            "required": ["city", "day"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "search_places",
        "description": "도시, 실내 필요 여부, 선호 활동을 기준으로 장소 후보를 찾습니다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "도시 이름. 예: 서울",
                },
                "indoor_required": {
                    "type": "boolean",
                    "description": "비가 오거나 실내 활동이 필요하면 true",
                },
                "preference": {
                    "type": "string",
                    "description": "선호 활동. 예: 문화, 카페, 쇼핑, 산책",
                },
            },
            "required": ["city", "indoor_required", "preference"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "estimate_total_cost",
        "description": "장소 이름과 인원수를 입력받아 총 예상 비용을 계산합니다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "place_name": {
                    "type": "string",
                    "description": "장소 이름",
                },
                "people": {
                    "type": "integer",
                    "description": "방문 인원수",
                },
            },
            "required": ["place_name", "people"],
            "additionalProperties": False,
        },
    },
]


# ------------------------------------------------------------
# 4. Tool Dispatcher
# ------------------------------------------------------------

def call_tool(name: str, args: dict) -> dict:
    """모델이 요청한 tool name을 실제 Python 함수에 연결합니다."""
    if name == "get_weather":
        return get_weather(**args)

    if name == "search_places":
        return search_places(**args)

    if name == "estimate_total_cost":
        return estimate_total_cost(**args)

    return {
        "error": f"Unknown tool: {name}",
    }


# ------------------------------------------------------------
# 5. ReAct Agent Loop  (Thought 출력이 추가된 버전)
# ------------------------------------------------------------
# 이론적 ReAct 사이클 그대로:
#
#   Thought      -> 다음에 뭘 할지 추론 (이 버전에서는 콘솔에 직접 출력됨)
#   Action       -> function_call 실행
#   Observation  -> 실행 결과 관찰
#   (반복) -> 더 필요한 정보가 없으면 최종 답변
#
# [보완한 부분]
# - INSTRUCTIONS에 "행동 전에 Thought: 한 줄을 반드시 남긴다"는 규칙을 추가했습니다.
# - Responses API에서 같은 턴에 (자연어 메시지 + function_call)이 함께 오면,
#   response.output_text에는 메시지 쪽 텍스트만 모여서 돌아옵니다.
#   즉 function_call이 있는 턴의 output_text == 모델이 남긴 Thought 문장이 됩니다.
# - 최종 답변 턴(= function_call이 없는 턴)에는 Thought가 따로 없으므로,
#   분기를 먼저 나눠서 Thought와 최종 답변이 중복 출력되지 않게 했습니다.

INSTRUCTIONS = """
너는 ReAct 스타일의 주말 나들이 추천 에이전트다.

목표:
- 사용자의 요청을 해결하기 위해 필요한 정보를 직접 추측하지 말고 도구로 확인한다.
- 날씨가 조건에 영향을 주면 반드시 get_weather를 먼저 사용한다.
- 장소 후보가 필요하면 search_places를 사용한다.
- 비용/예산 판단이 필요하면 estimate_total_cost를 사용한다.
- 도구 호출은 한 번에 하나씩만 한다.
- 최종 답변은 한국어로 작성한다.

중요 — Thought 규칙:
- 도구를 호출하기 전에 반드시 "Thought: "로 시작하는 한 문장을 먼저 작성한다.
  예: "Thought: 토요일 날씨를 먼저 확인해야 실내/야외 여부를 판단할 수 있다."
- 더 이상 도구가 필요 없어 최종 답변을 낼 때는 Thought 문장 없이 최종 답변 형식만 작성한다.

최종 답변 형식:
1. 추천 장소
2. 추천 이유
3. 날씨 고려
4. 예상 비용
5. 예산 충족 여부
6. 대안 후보
"""


def run_react_agent(user_question: str, max_steps: int = 8) -> str:
    input_items = [
        {
            "role": "user",
            "content": user_question,
        }
    ]

    for step in range(1, max_steps + 1):
        print(f"\n========== STEP {step} ==========")

        response = client.responses.create(
            model=MODEL,
            instructions=INSTRUCTIONS,
            input=input_items,
            tools=TOOLS,
            temperature=0.2,
            tool_choice="auto",
            # 초보자 학습용이므로 한 번에 하나의 도구만 호출하게 합니다.
            # ReAct 흐름을 보기 훨씬 쉽습니다.
            parallel_tool_calls=False,
        )

        # 다음 턴에 모델의 직전 출력도 같이 넘겨야 합니다.
        input_items += response.output

        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        # 더 이상 tool call이 없으면 최종 답변입니다.
        # (이 턴에는 Thought가 따로 없으므로 바로 최종 답변만 출력합니다.)
        if not function_calls:
            print("\n========== FINAL ANSWER ==========")
            print(response.output_text)
            return response.output_text

        # [보완된 부분] Thought 출력
        # function_call이 있는 턴에 모델이 같이 남긴 "Thought: ..." 문장을 보여줍니다.
        thought_text = response.output_text.strip()
        if thought_text:
            print(f"[Thought] {thought_text}")

        # parallel_tool_calls=False라 보통 하나만 오지만,
        # 안전하게 여러 개도 처리 가능하게 둡니다.
        for function_call in function_calls:
            tool_name = function_call.name
            tool_args = json.loads(function_call.arguments)

            print(f"[Action] {tool_name}")
            print(f"[Action Input] {json.dumps(tool_args, ensure_ascii=False, indent=2)}")

            tool_result = call_tool(tool_name, tool_args)

            print("[Observation]")
            print(json.dumps(tool_result, ensure_ascii=False, indent=2))

            input_items.append({
                "type": "function_call_output",
                "call_id": function_call.call_id,
                "output": json.dumps(tool_result, ensure_ascii=False),
            })

    raise RuntimeError("최대 step 수를 초과했습니다. 도구 루프가 끝나지 않았습니다.")




In [12]:
# ------------------------------------------------------------
# 6. Test
# ------------------------------------------------------------
question = """
    이번 토요일에 서울에서 친구랑 2명이 갈 만한 곳 추천해줘.
    비가 오면 실내였으면 좋겠고, 전체 예산은 6만원 이하였으면 해.
    나는 문화/전시 쪽을 선호해.
    """

run_react_agent(question)


========== STEP 1 ==========
[Action] get_weather
[Action Input] {
  "city": "서울",
  "day": "토요일"
}
[Observation]
{
  "city": "서울",
  "day": "토요일",
  "condition": "비",
  "temperature_c": 24,
  "summary": "오후에 비가 올 가능성이 높습니다. 실내 활동을 추천합니다."
}

========== STEP 2 ==========
[Action] search_places
[Action Input] {
  "city": "서울",
  "indoor_required": true,
  "preference": "문화"
}
[Observation]
{
  "city": "서울",
  "indoor_required": true,
  "preference": "문화",
  "places": [
    {
      "city": "서울",
      "name": "국립중앙박물관",
      "type": "문화",
      "indoor": true,
      "price_per_person": 0,
      "duration_hours": 2.5,
      "description": "비 오는 날에도 편하게 볼 수 있는 대형 실내 전시 공간입니다.",
      "match_score": 3
    },
    {
      "city": "서울",
      "name": "성수동 카페거리",
      "type": "카페",
      "indoor": true,
      "price_per_person": 15000,
      "duration_hours": 2,
      "description": "카페와 편집숍을 함께 둘러보기 좋습니다.",
      "match_score": 1
    },
    {
      "city": "서울",
      "name": "잠실 롯데월드몰",
  

'1. 추천 장소: 국립중앙박물관\n2. 추천 이유: 비 오는 날에도 편하게 볼 수 있는 대형 실내 전시 공간으로, 다양한 문화 전시를 즐길 수 있습니다.\n3. 날씨 고려: 토요일에 비가 올 가능성이 높아 실내 활동이 적합합니다.\n4. 예상 비용: 0원 (입장료가 없습니다)\n5. 예산 충족 여부: 예산 6만원 이하로 충분히 충족됩니다.\n6. 대안 후보: 성수동 카페거리 (1인당 15,000원) 또는 잠실 롯데월드몰 (1인당 20,000원)도 고려할 수 있습니다.'

**📝 정리** — 비 오는 토요일 → `get_weather` → `search_places`(실내·문화) → `estimate_total_cost` 순서로 도구를 하나씩 불러 국립중앙박물관(0원)을 추천했다. 이 실행 출력에는 `[Thought]` 줄이 없고, 다음 셀부터 보인다.

In [13]:
run_react_agent("이번 일요일에 서울에서 친구랑 2명이 갈 만한 곳 추천해줘. ")


========== STEP 1 ==========
[Thought] Thought: 일요일 날씨를 먼저 확인해야 실내/야외 여부를 판단할 수 있다.
[Action] get_weather
[Action Input] {
  "city": "서울",
  "day": "일요일"
}
[Observation]
{
  "city": "서울",
  "day": "일요일",
  "condition": "맑음",
  "temperature_c": 28,
  "summary": "대체로 맑고 야외 활동하기 좋습니다."
}

========== STEP 2 ==========

========== FINAL ANSWER ==========
1. 추천 장소: 한강공원
2. 추천 이유: 대체로 맑고 야외 활동하기 좋은 날씨에 친구와 함께 산책이나 자전거 타기를 즐길 수 있습니다.
3. 날씨 고려: 일요일은 맑고 기온이 28도여서 야외 활동에 적합합니다.
4. 예상 비용: 자전거 대여 비용 약 5,000원 (1인 기준, 2명 총 10,000원)
5. 예산 충족 여부: 10,000원은 일반적인 주말 나들이 예산에 적합합니다.
6. 대안 후보: 서울숲 (산책 및 피크닉 가능)


'1. 추천 장소: 한강공원\n2. 추천 이유: 대체로 맑고 야외 활동하기 좋은 날씨에 친구와 함께 산책이나 자전거 타기를 즐길 수 있습니다.\n3. 날씨 고려: 일요일은 맑고 기온이 28도여서 야외 활동에 적합합니다.\n4. 예상 비용: 자전거 대여 비용 약 5,000원 (1인 기준, 2명 총 10,000원)\n5. 예산 충족 여부: 10,000원은 일반적인 주말 나들이 예산에 적합합니다.\n6. 대안 후보: 서울숲 (산책 및 피크닉 가능)'

**📝 정리** — **값을 지어냈다**: 날씨만 확인하고 장소·비용 도구를 건너뛴 채 "자전거 대여 약 5,000원"이라고 답했다(DB의 한강공원은 1인 10,000원). 지시문에 "추측하지 말고 도구로 확인"이라고 써도 모델이 건너뛸 수 있다.

In [14]:
run_react_agent("이번 일요일에 서울에서 친구랑 2명이 갈 만한 곳 추천해줘. 근데 예산이 좀 부족해... 둘이서 10000원으로 즐길 수 있는 곳을 추천해줘")


========== STEP 1 ==========
[Thought] Thought: 일요일 날씨를 먼저 확인해야 실내/야외 여부를 판단할 수 있다.
[Action] get_weather
[Action Input] {
  "city": "서울",
  "day": "일요일"
}
[Observation]
{
  "city": "서울",
  "day": "일요일",
  "condition": "맑음",
  "temperature_c": 28,
  "summary": "대체로 맑고 야외 활동하기 좋습니다."
}

========== STEP 2 ==========
[Action] search_places
[Action Input] {
  "city": "서울",
  "indoor_required": false,
  "preference": "산책"
}
[Observation]
{
  "city": "서울",
  "indoor_required": false,
  "preference": "산책",
  "places": [
    {
      "city": "서울",
      "name": "서울숲",
      "type": "산책",
      "indoor": false,
      "price_per_person": 0,
      "duration_hours": 2,
      "description": "날씨가 좋을 때 산책하기 좋은 야외 공간입니다.",
      "match_score": 3
    },
    {
      "city": "서울",
      "name": "한강공원",
      "type": "산책",
      "indoor": false,
      "price_per_person": 10000,
      "duration_hours": 2,
      "description": "돗자리, 간식, 자전거 등을 즐길 수 있는 야외 코스입니다.",
      "match_score": 2
    },
    {
      "ci

'1. 추천 장소: 한강공원\n2. 추천 이유: 돗자리와 간식을 가져가면 야외에서 편안하게 시간을 보낼 수 있는 좋은 장소입니다.\n3. 날씨 고려: 일요일은 대체로 맑고 야외 활동하기 좋은 날씨입니다.\n4. 예상 비용: 총 20,000원 (2명 기준, 1인 10,000원)\n5. 예산 충족 여부: 예산 10,000원 초과\n6. 대안 후보: 서울숲 (무료로 산책 가능)'

**📝 정리** — **결론이 틀렸다**: 비용 도구로 한강공원 2명 20,000원을 확인하고 "예산 초과"라고 스스로 적었는데도 1순위로 추천했다. 대안의 서울숲(0원)이 조건에 맞는다. 예산처럼 숫자로 확인할 수 있는 조건은 코드로 다시 검사하는 게 안전하다.

### ReAct 소스(3)

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [15]:
"""
목적:
    "도구 결과가 원하는 결과가 아니면 다시 시도하는 ReAct 루프"를 확인하는 최소 예제입니다.

핵심 시나리오:
    사용자: "7명 회의실 예약해줘. 먼저 Alpha 회의실로 시도해."
    1) 모델이 reserve_room("Alpha") 호출
    2) 도구 결과: Alpha는 unavailable
    3) 모델이 실패 Observation을 보고 search_available_rooms 호출
    4) 모델이 대체 회의실 Beta를 골라 reserve_room("Beta") 재호출
    5) 도구 결과: success
    6) 모델이 최종 답변
"""

from __future__ import annotations

import json
import os
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI


# ---------------------------------------------------------------------
# 1. 가짜 회의실 DB
# ---------------------------------------------------------------------

ROOM_DB = {
    "Alpha": {
        "capacity": 10,
        "available": False,
        "reason": "이미 예약되어 있습니다.",
    },
    "Beta": {
        "capacity": 8,
        "available": True,
        "reason": None,
    },
    "Gamma": {
        "capacity": 4,
        "available": True,
        "reason": None,
    },
    "Delta": {
        "capacity": 12,
        "available": True,
        "reason": None,
    },
}


# ---------------------------------------------------------------------
# 2. 로컬 도구 함수
# ---------------------------------------------------------------------

def reserve_room(room_name: str, date: str, start_time: str, duration_minutes: int, people: int) -> dict[str, Any]:
    """
    회의실 예약을 시도합니다.
    성공하면 success=True, 실패하면 success=False를 반환합니다.
    """

    room = ROOM_DB.get(room_name)

    if room is None:
        return {
            "success": False,
            "room_name": room_name,
            "status": "not_found",
            "message": "존재하지 않는 회의실입니다.",
        }

    if room["capacity"] < people:
        return {
            "success": False,
            "room_name": room_name,
            "status": "too_small",
            "capacity": room["capacity"],
            "people": people,
            "message": f"{people}명을 수용하기에는 회의실이 작습니다.",
        }

    if not room["available"]:
        return {
            "success": False,
            "room_name": room_name,
            "status": "unavailable",
            "message": room["reason"],
        }

    return {
        "success": True,
        "room_name": room_name,
        "status": "reserved",
        "date": date,
        "start_time": start_time,
        "duration_minutes": duration_minutes,
        "people": people,
        "message": "예약이 완료되었습니다.",
    }


def search_available_rooms(date: str, start_time: str, duration_minutes: int, people: int) -> dict[str, Any]:
    """
    조건에 맞는 대체 회의실을 검색합니다.
    """

    candidates = []

    for room_name, room in ROOM_DB.items():
        if room["available"] and room["capacity"] >= people:
            candidates.append(
                {
                    "room_name": room_name,
                    "capacity": room["capacity"],
                }
            )

    return {
        "date": date,
        "start_time": start_time,
        "duration_minutes": duration_minutes,
        "people": people,
        "candidates": candidates,
    }


# ---------------------------------------------------------------------
# 3. OpenAI에 전달할 Tool Schema
# ---------------------------------------------------------------------

TOOLS = [
    {
        "type": "function",
        "name": "reserve_room",
        "description": "지정한 회의실을 예약합니다. 예약 실패 시 실패 이유를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "room_name": {
                    "type": "string",
                    "description": "예약할 회의실 이름. 예: Alpha, Beta, Gamma, Delta",
                },
                "date": {
                    "type": "string",
                    "description": "예약 날짜. 예: 2026-07-01",
                },
                "start_time": {
                    "type": "string",
                    "description": "시작 시간. 예: 10:00",
                },
                "duration_minutes": {
                    "type": "integer",
                    "description": "회의 시간. 단위: 분",
                },
                "people": {
                    "type": "integer",
                    "description": "참석 인원 수",
                },
            },
            "required": ["room_name", "date", "start_time", "duration_minutes", "people"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "search_available_rooms",
        "description": "조건에 맞는 예약 가능한 회의실 후보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "예약 날짜. 예: 2026-07-01",
                },
                "start_time": {
                    "type": "string",
                    "description": "시작 시간. 예: 10:00",
                },
                "duration_minutes": {
                    "type": "integer",
                    "description": "회의 시간. 단위: 분",
                },
                "people": {
                    "type": "integer",
                    "description": "참석 인원 수",
                },
            },
            "required": ["date", "start_time", "duration_minutes", "people"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


# ---------------------------------------------------------------------
# 4. 모델이 요청한 tool name을 실제 파이썬 함수로 연결
# ---------------------------------------------------------------------

def call_tool(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    if name == "reserve_room":
        return reserve_room(**arguments)

    if name == "search_available_rooms":
        return search_available_rooms(**arguments)

    return {
        "success": False,
        "error": f"알 수 없는 도구입니다: {name}",
    }


# ---------------------------------------------------------------------
# 5. ReAct 반복 루프
# ---------------------------------------------------------------------

def run_react_retry(user_prompt: str, max_steps: int = 6) -> str:
    load_dotenv()

    client = OpenAI()
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

    input_items: list[Any] = [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

    print("\n==============================")
    print("ReAct retry loop check start")
    print("==============================")
    print(f"model: {model}")
    print(f"user: {user_prompt}")

    for step in range(1, max_steps + 1):
        print(f"\n--- STEP {step}: model thinking/action ---")

        response = client.responses.create(
            model=model,
            instructions=(
                "너는 회의실 예약 에이전트다. "
                "반드시 도구를 사용해서 예약을 시도해야 한다. "
                "처음 요청한 회의실 예약에 실패하면, 실패했다고 바로 끝내지 말고 "
                "예약 가능한 대체 회의실을 검색한 뒤 다시 예약을 시도하라. "
                "예약 성공 Observation을 받은 뒤에만 최종 답변하라. "
                "최종 답변은 한국어로 짧게 작성하라."
            ),
            tools=TOOLS,
            input=input_items,
        )

        # 모델의 이번 출력 기록을 다음 입력에 유지합니다.
        # 여기에는 function_call도 포함됩니다.
        input_items += response.output

        function_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        # function_call이 없으면 모델이 최종 답변을 낸 것입니다.
        if not function_calls:
            final_text = response.output_text or ""
            print("\n--- FINAL ANSWER ---")
            print(final_text)
            return final_text

        # 모델이 요청한 tool call들을 실행합니다.
        for call in function_calls:
            tool_name = call.name
            tool_args = json.loads(call.arguments)

            print(f"\nAction: {tool_name}")
            print(f"Arguments: {json.dumps(tool_args, ensure_ascii=False)}")

            tool_result = call_tool(tool_name, tool_args)

            print(f"Observation: {json.dumps(tool_result, ensure_ascii=False)}")

            # 실행 결과를 다시 모델에게 전달합니다.
            # call_id로 어떤 function_call의 결과인지 연결됩니다.
            input_items.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(tool_result, ensure_ascii=False),
                }
            )

    raise RuntimeError(f"max_steps={max_steps} 안에 완료하지 못했습니다.")


if __name__ == "__main__":
    prompt = (
        "2026-09-12 10:00에 7명 회의를 60분 동안 예약해줘. "
        "먼저 Alpha 회의실로 예약을 시도해줘. "
        "Alpha가 안 되면 가능한 다른 회의실을 찾아서 다시 예약해줘."
    )

    run_react_retry(prompt)



ReAct retry loop check start
model: gpt-4o-mini
user: 2026-09-12 10:00에 7명 회의를 60분 동안 예약해줘. 먼저 Alpha 회의실로 예약을 시도해줘. Alpha가 안 되면 가능한 다른 회의실을 찾아서 다시 예약해줘.

--- STEP 1: model thinking/action ---

Action: reserve_room
Arguments: {"room_name": "Alpha", "date": "2026-09-12", "start_time": "10:00", "duration_minutes": 60, "people": 7}
Observation: {"success": false, "room_name": "Alpha", "status": "unavailable", "message": "이미 예약되어 있습니다."}

--- STEP 2: model thinking/action ---

Action: search_available_rooms
Arguments: {"date": "2026-09-12", "start_time": "10:00", "duration_minutes": 60, "people": 7}
Observation: {"date": "2026-09-12", "start_time": "10:00", "duration_minutes": 60, "people": 7, "candidates": [{"room_name": "Beta", "capacity": 8}, {"room_name": "Delta", "capacity": 12}]}

--- STEP 3: model thinking/action ---

Action: reserve_room
Arguments: {"room_name": "Beta", "date": "2026-09-12", "start_time": "10:00", "duration_minutes": 60, "people": 7}
Observation: {"success": true, 

**📝 정리** — Alpha 예약 실패(이미 예약됨) → 대체 회의실 검색 → Beta 예약 성공. 실패한 Observation을 보고 다른 행동을 골랐다. 도구 명세의 `"strict": True` + `"additionalProperties": False`로 인자 형식을 강제했다.

In [5]:
# ask_human: 그냥 Enter = 승인, 아무 글자나 입력하면 거부(+ 수정 지시)로 처리된다
"""
Human-in-the-loop ReAct 에이전트 예제: 야식 주문 에이전트(봇)🍗

- Python 3.12 / openai SDK / gpt-4o-mini
- ReAct = Thought(생각) → Action(도구 호출) → Observation(결과) 반복
- Human in the loop = '돈이 나가는 행동(주문)'만 사람이 승인/거부/수정 지시

설치:  pip install openai
실행:  export OPENAI_API_KEY=sk-...   
"""
import json
from openai import OpenAI

client = OpenAI()  
MODEL = "gpt-4o-mini"

# 👇 False 로 바꿔서 실행해 보면 사람이 없을 때 에이전트가 얼마나 폭주(?)하는지 비교할 수 있음
HUMAN_IN_THE_LOOP = True

# ---------------------------------------------------------------
# 1. 가짜 세상: 메뉴판과 지갑
# ---------------------------------------------------------------
MENU = [
    {"name": "양념치킨", "price": 19000, "tag": ["매움", "매운"]},
    {"name": "후라이드치킨", "price": 18000, "tag": ["고소", "고소한", "고기"]},
    {"name": "불닭볶음면", "price": 2500, "tag": ["매움", "매운"]},
    {"name": "떡볶이", "price": 5000, "tag": ["매움", "매운"]},
    {"name": "냉면", "price": 7000, "tag": ["시원", "시원한"]},
    {"name": "김밥", "price": 3500, "tag": ["든든", "간단"]},
]
wallet = {"balance": 20000}


# ---------------------------------------------------------------
# 2. 도구(Action) 정의
#    - search_menu, check_budget : 안전한 조회 → 자동 실행
#    - order_food                : 돈이 나가는 위험한 행동 → 사람 승인 필요
# ---------------------------------------------------------------
def search_menu(keyword: str) -> str:
    found = [m for m in MENU if keyword in m["name"] or keyword in m["tag"]]
    return json.dumps(found or MENU, ensure_ascii=False)  # 없으면 전체 메뉴


def check_budget() -> str:
    return f"남은 잔액: {wallet['balance']}원"


def order_food(name: str, quantity: int = 1) -> str:
    item = next((m for m in MENU if m["name"] == name), None)
    if item is None:
        return f"'{name}' 은(는) 메뉴에 없습니다."
    total = item["price"] * quantity
    if total > wallet["balance"]:
        return f"잔액 부족! 필요 {total}원 / 잔액 {wallet['balance']}원"
    wallet["balance"] -= total
    return f"주문 완료: {name} x{quantity} ({total}원 결제, 잔액 {wallet['balance']}원)"


TOOLS_IMPL = {"search_menu": search_menu, "check_budget": check_budget, "order_food": order_food}
NEEDS_APPROVAL = {"order_food"}  # 👈 사람 승인이 필요한 도구 목록

TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "search_menu", "description": "메뉴를 키워드(이름/맛)로 검색합니다.",
        "parameters": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]}}},
    {"type": "function", "function": {
        "name": "check_budget", "description": "현재 잔액을 확인합니다.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "order_food", "description": "음식을 실제로 주문(결제)합니다.",
        "parameters": {"type": "object", "properties": {
            "name": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["name"]}}},
]

SYSTEM_PROMPT = """너는 야식 주문 도우미 에이전트입니다. ReAct 방식으로 일합니다.
- 도구를 호출하기 전에 반드시 'Thought: ...' 한 줄로 왜 그 행동을 하는지 먼저 적습니다.
- 주문 전에 메뉴 검색과 잔액 확인을 먼저 합니다.
- 주문이 사람에게 거부되면, 사람의 사유를 반영해서 다른 메뉴를 다시 제안/주문합니다.
- 주문이 끝나면 'Final Answer: ...' 로 한 줄 요약합니다."""


# ---------------------------------------------------------------
# 3. Human in the loop: 위험한 행동 직전에 사람에게 묻는 관문
# ---------------------------------------------------------------
def ask_human(name: str, args: dict) -> tuple[bool, str]:
    print(f"\n🙋 [승인 요청] {name}({args})")
    answer = input("   승인 = Enter / 거부·수정 지시 = 하고 싶은 말 입력 > ").strip()
    return answer == "", answer


# ---------------------------------------------------------------
# 4. ReAct 루프
# ---------------------------------------------------------------
def run_agent(user_request: str, max_steps: int = 8) -> None:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_request},
    ]

    for step in range(1, max_steps + 1):
        res = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS_SPEC,
            parallel_tool_calls=False,  # 한 번에 하나씩 → 승인 흐름이 또렷해짐
        )
        msg = res.choices[0].message
        messages.append(msg)

        if msg.content:  # Thought / Final Answer
            print(f"\n💭 {msg.content}")

        if not msg.tool_calls:  # 도구 호출이 없으면 종료
            return

        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"🔧 Action: {name}({args})")

            # ✋ 위험한 도구는 사람에게 먼저 물어본다
            if HUMAN_IN_THE_LOOP and name in NEEDS_APPROVAL:
                approved, feedback = ask_human(name, args)
                if not approved:
                    observation = f"사람이 주문을 거부했습니다. 사유/지시: {feedback}"
                    print(f"👀 Observation: {observation}")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": observation})
                    continue

            observation = TOOLS_IMPL[name](**args)
            print(f"👀 Observation: {observation}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": observation})

    print("⚠️ 최대 단계 도달: 에이전트를 중단합니다.")



In [6]:
run_agent("배고파... 매운 걸로 알아서 야식 시켜줘!")



💭 Thought: 매운 메뉴를 찾기 위해 메뉴 검색을 먼저 진행하겠습니다.  
Let's search for spicy food options.
🔧 Action: search_menu({'keyword': '매운'})
👀 Observation: [{"name": "양념치킨", "price": 19000, "tag": ["매움", "매운"]}, {"name": "불닭볶음면", "price": 2500, "tag": ["매움", "매운"]}, {"name": "떡볶이", "price": 5000, "tag": ["매움", "매운"]}]
🔧 Action: check_budget({})
👀 Observation: 남은 잔액: 20000원

💭 Thought: 현재 잔액이 20,000원이므로 매운 메뉴를 주문할 수 있습니다. 양념치킨, 불닭볶음면, 떡볶이 중에서 선택할 수 있습니다. 양념치킨을 주문해보겠습니다.  
Let's order "양념치킨".
🔧 Action: order_food({'name': '양념치킨'})

🙋 [승인 요청] order_food({'name': '양념치킨'})
👀 Observation: 사람이 주문을 거부했습니다. 사유/지시: 다른 건 없어요?

💭 Thought: 양념치킨이 거부되었으므로 다른 매운 메뉴 중에서 선택하겠습니다. 불닭볶음면과 떡볶이 중 한번 더 제안해보겠습니다. 불닭볶음면을 주문해보겠습니다.  
Let's order "불닭볶음면".
🔧 Action: order_food({'name': '불닭볶음면'})

🙋 [승인 요청] order_food({'name': '불닭볶음면'})
👀 Observation: 사람이 주문을 거부했습니다. 사유/지시: 불닭볶음면 좋아요

💭 Thought: 불닭볶음면이 마음에 드신다고 하니, 이 메뉴를 주문해보겠습니다.  
Let's go ahead and order "불닭볶음면" again.
🔧 Action: order_food({'name': '불닭볶음면'})

🙋 [승인 요청] order_fo

**📝 정리** — 주문(`order_food`) 직전에만 사람에게 물었다. "불닭볶음면 좋아요"라고 입력했더니 **거부로 처리**되어(Enter만 승인) 에이전트가 같은 주문을 한 번 더 요청했다. 승인 UI는 예/아니오를 분명히 받는 편이 좋다.

In [7]:
run_agent("시원한거 먹고싶다")



💭 Thought: '시원한' 음식을 먹고 싶다는 요청에 따라, 관련된 메뉴를 검색하여 적합한 음식을 찾아야 한다. 
메뉴를 검색하겠습니다. 

{"keyword":"시원한"}
🔧 Action: search_menu({'keyword': '시원한'})
👀 Observation: [{"name": "냉면", "price": 7000, "tag": ["시원", "시원한"]}]

💭 Thought: 메뉴에서 '냉면'이 시원한 음식으로 확인되었으므로, 주문 전에 잔액을 확인해야 한다.
잔액을 확인하겠습니다. 

{}
🔧 Action: check_budget({})
👀 Observation: 남은 잔액: 17500원

💭 Thought: 잔액이 17,500원이므로 냉면(7,000원)을 주문할 수 있다. 
냉면을 주문하겠습니다. 

{"name":"냉면","quantity":1}
🔧 Action: order_food({'name': '냉면', 'quantity': 1})

🙋 [승인 요청] order_food({'name': '냉면', 'quantity': 1})
👀 Observation: 주문 완료: 냉면 x1 (7000원 결제, 잔액 10500원)

💭 Final Answer: 냉면 1개를 주문했습니다. (잔액 10,500원)


In [8]:
run_agent("고기 먹고싶다")




💭 Thought: 고기를 먹고 싶다고 하셨으니, 먼저 고기 메뉴를 검색해 보겠습니다.  
{"keyword": "고기"}
🔧 Action: search_menu({'keyword': '고기'})
👀 Observation: [{"name": "후라이드치킨", "price": 18000, "tag": ["고소", "고소한", "고기"]}]

💭 Thought: 후라이드치킨이 검색되었습니다. 다음으로 현재 잔액을 확인해 보겠습니다.  
{"keyword": ""}
🔧 Action: check_budget({})
👀 Observation: 남은 잔액: 10500원

💭 Thought: 현재 잔액이 10,500원으로 후라이드치킨(18,000원)을 주문하기에는 부족합니다. 따라서, 다른 고기 메뉴를 제안해야 합니다.  
{"keyword": "소고기"}
🔧 Action: search_menu({'keyword': '소고기'})
👀 Observation: [{"name": "양념치킨", "price": 19000, "tag": ["매움", "매운"]}, {"name": "후라이드치킨", "price": 18000, "tag": ["고소", "고소한", "고기"]}, {"name": "불닭볶음면", "price": 2500, "tag": ["매움", "매운"]}, {"name": "떡볶이", "price": 5000, "tag": ["매움", "매운"]}, {"name": "냉면", "price": 7000, "tag": ["시원", "시원한"]}, {"name": "김밥", "price": 3500, "tag": ["든든", "간단"]}]

💭 Thought: 다른 메뉴를 검색해본 결과, 불닭볶음면(2,500원), 떡볶이(5,000원), 냉면(7,000원), 김밥(3,500원)이 있습니다. 이 중에서 주문할 수 있는 메뉴를 제안하겠습니다.  
주문할 수 있는 메뉴는 불닭볶음면, 떡볶이, 냉면, 김밥입니다. 최소한의 고기의 맛을 느낄 수 있는 메뉴인 떡볶이를 추천드립니다

**📝 정리** — '소고기' 검색 결과가 없자 `search_menu`가 전체 메뉴를 돌려줬고(`found or MENU`), 에이전트는 떡볶이를 "고기 맛"이라며 추천했다. 또 `run_agent`마다 대화 기록이 새로 시작해서 앞에서 냉면을 주문한 것을 모르고 다시 제안했다(잔액 `wallet`만 이어짐).

<hr style="background-color:red; height:3px">

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### <span style='color:blue; font-weight:bold'>다중 LLM 활용 그리고 ReAct(생각,행동)</span> 에 대해 학습했습니다.

## 🔁 복습 질문

<details>
<summary>1. 병렬 다중 LLM 패턴에서 <code>ThreadPoolExecutor</code>로 얻는 것은?</summary>

응답을 기다리는 시간을 겹쳐서, 세 모델을 하나씩 부를 때보다 전체 시간이 짧아진다.
</details>

<details>
<summary>2. Gemini 답이 에러였는데 종합 리포트가 만들어진 이유와 막는 방법은?</summary>

`ask_gemini`가 예외를 문자열로 돌려줘서 다음 단계가 에러인지 몰랐다. 성공 여부를 따로 표시하고 실패한 답은 종합에서 뺀다.
</details>

<details>
<summary>3. ReAct에서 Observation이 중요한 이유는?</summary>

행동의 실제 결과를 보고 다음 행동을 정하기 때문이다. 실패하면 다시 시도하거나 다른 도구를 고를 수 있다.
</details>

<details>
<summary>4. Human-in-the-loop에서 사람 승인을 받을 도구는?</summary>

결제·삭제·전송처럼 되돌릴 수 없거나 비용이 드는 행동. 조회 도구는 자동 실행해도 된다.
</details>
